# Evaluation: Fine-Tuned Mistral 7B vs GPT-4o-mini

### 1. Environment Setup

Install dependencies and configure environment variables for Hugging Face and Azure OpenAI.

In [ ]:
!pip install -q peft trl bitsandbytes wandb sentencepiece pydantic rapidfuzz openai python-dotenv

import os, sys, json, torch, gc
from tqdm import tqdm

# Check GPU — P100 is not compatible with bitsandbytes
gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
print(f"GPU: {gpu_name[0]}")
if "P100" in gpu_name[0]:
    raise RuntimeError("Got P100 (sm_60) — not compatible with bitsandbytes. Restart session to get T4.")

### 2. Configuration

Set API keys and model configuration.
⚠️ In production, use environment variables or secret managers instead of hardcoding.

In [ ]:
import os, sys, json, subprocess
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['AZURE_OPENAI_API_KEY'] = 'YOUR_AZURE_KEY'
os.environ['AZURE_OPENAI_ENDPOINT'] = 'YOUR_AZURE_ENDPOINT'
os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-4o-mini'
os.environ['AZURE_OPENAI_API_VERSION'] = '2024-12-01-preview'

### 3. Load Project Repository

Clone the fine-tuning project and add it to the Python path.

In [ ]:
!git clone https://github.com/Pushparaj13811/invoice-extraction-mistral-7b-fine-tuning.git
sys.path.insert(0, '/kaggle/working/invoice-extraction-mistral-7b-fine-tuning')

### 4. Load Evaluation Data

Load evaluation dataset and parse ground truth into structured `Invoice` objects.

In [ ]:
from src.data.format import load_jsonl
from src.data.schema import Invoice

eval_data = load_jsonl('/kaggle/input/datasets/pushparajmehta/invoice-extraction-dataset/eval.jsonl')
gold_invoices = [Invoice.model_validate_json(ex['output']) for ex in eval_data]

print(f'Eval samples: {len(eval_data)}')

### 5. Load Fine-Tuned Model

Load base model + LoRA adapter.

In [ ]:
from src.evaluation.evaluate import load_finetuned_model

ADAPTER_PATH = '/kaggle/input/notebooks/pushparajmehta/invoice-extraction-training-v2/final_adapter'

model, tokenizer = load_finetuned_model('mistralai/Mistral-7B-v0.3', ADAPTER_PATH)
model.eval()

print('Model loaded')

### 6. Prompt Formatting & Utilities

Define prompt structure and helper functions for:
- output cleaning
- safe JSON parsing into schema

In [ ]:
def format_prompt(example):
    return f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
"""

def clean_output(prompt, generated):
    marker = "### Response:\n"
    idx = generated.find(marker)
    if idx >= 0:
        return generated[idx + len(marker):].strip()
    return generated[len(prompt):].strip()

def safe_parse(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines)
    try:
        data = json.loads(text)
        return Invoice.model_validate(data)
    except:
        return None

### 7. Fine-Tuned Model Inference (Batched)

Run efficient batched inference and convert outputs into structured `Invoice` objects.

In [ ]:
def run_batch(model, tokenizer, prompts, max_new_tokens=256):
    with torch.no_grad():
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

        return tokenizer.batch_decode(outputs, skip_special_tokens=True)


def batched_inference(model, tokenizer, eval_data, batch_size=8):
    final_predictions = []

    for i in tqdm(range(0, len(eval_data), batch_size), desc="FT Inference"):
        batch = eval_data[i:i+batch_size]

        prompts = [format_prompt(x) for x in batch]

        raw_outputs = run_batch(model, tokenizer, prompts)

        cleaned = [
            clean_output(inp, out)
            for inp, out in zip(prompts, raw_outputs)
        ]

        parsed = [safe_parse(x) for x in cleaned]

        final_predictions.extend(parsed)

    return final_predictions

### 8. Execute Fine-Tuned Model Evaluation

In [ ]:
ft_predictions = batched_inference(model, tokenizer, eval_data, batch_size=8)

ft_parsed = sum(p is not None for p in ft_predictions)

print(f'Parsed: {ft_parsed}/{len(ft_predictions)} ({ft_parsed/len(ft_predictions)*100:.1f}%)')

### 9. Cleanup GPU Memory

Release GPU resources before running baseline model.

In [ ]:
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f'GPU freed: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB available')

### 10. Baseline Evaluation (GPT-4o-mini)

Run baseline using GPT-4o-mini for comparison.

In [ ]:
from src.evaluation.baseline import run_baseline

baseline_predictions = run_baseline(eval_data)

bl_parsed = sum(p is not None for p in baseline_predictions)

print(f'Parsed: {bl_parsed}/{len(baseline_predictions)} ({bl_parsed/len(baseline_predictions)*100:.1f}%)')

### 11. Compute Metrics

Evaluate both models against ground truth.

In [ ]:
from src.evaluation.evaluate import aggregate_metrics, generate_report

ft_metrics = aggregate_metrics(ft_predictions, gold_invoices)
baseline_metrics = aggregate_metrics(baseline_predictions, gold_invoices)

### 12. Model Comparison

In [ ]:
print(f"{'Metric':<25} {'Fine-Tuned':>12} {'GPT-4o-mini':>12}")
print('-' * 52)

print(f"{'Overall Accuracy':<25} {ft_metrics['overall_accuracy']:>11.1%} {baseline_metrics['overall_accuracy']:>11.1%}")
print(f"{'JSON Parse Rate':<25} {ft_metrics['json_parse_success_rate']:>11.1%} {baseline_metrics['json_parse_success_rate']:>11.1%}")
print(f"{'Line Item Score':<25} {ft_metrics['line_item_score']:>11.1%} {baseline_metrics['line_item_score']:>11.1%}")

### 13. Per-Field Accuracy Analysis

In [ ]:
print(f"{'Field':<25} {'Fine-Tuned':>12} {'GPT-4o-mini':>12} {'Improvement':>12}")
print('-' * 64)

for field in sorted(set(list(ft_metrics['per_field'].keys()) + list(baseline_metrics['per_field'].keys()))):
    ft_val = ft_metrics['per_field'].get(field, 0)
    bl_val = baseline_metrics['per_field'].get(field, 0)

    imp = ((ft_val - bl_val) / bl_val * 100) if bl_val > 0 else 0

    print(f'{field:<25} {ft_val:>11.1%} {bl_val:>11.1%} {imp:>+11.1f}%')

### 14. Generate Evaluation Report

In [ ]:
report = generate_report(ft_metrics, baseline_metrics)

with open('/kaggle/working/evaluation_report.md', 'w') as f:
    f.write(report)

print('Report saved')

### 15. Save Results for Analysis

In [ ]:
results = {
    'ft_metrics': ft_metrics,
    'baseline_metrics': baseline_metrics,
    'ft_predictions': [p.model_dump() if p else None for p in ft_predictions],
    'baseline_predictions': [p.model_dump() if p else None for p in baseline_predictions],
}

with open('/kaggle/working/evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved')